<a href="https://colab.research.google.com/github/mkuznetsov2007-dot/ABOBA/blob/main/%D1%8D%D0%BB%D0%B5%D0%BA%D1%82%D1%80%D0%BE%D1%8D%D0%BD%D0%B5%D1%80%D0%B3%D0%B5%D1%82%D0%B8%D0%BA%D0%B0_%D0%BB%D0%B0%D0%B1%D0%B0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Подключение нужных для работы моделей
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import matplotlib.pyplot as plt

In [ ]:
# 2. Получение и чтение файла из облака
file_id = '1R-bDTTCsLjzy9NZG3bkzzeNnFyBuupGK'
url = f"https://drive.google.com/uc?export=download&id={file_id}"
# Чтение данных
df = pd.read_csv(url, sep=';', encoding='utf-8')


In [ ]:
# 3. Первые 5 строк фйла
df.head()

,Forecast,Station_A,Station_B,Station_C,Station_D
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 4. Последние 5 строк файла
df.tail()

,Forecast,Station_A,Station_B,Station_C,Station_D
2055,NaN,NaN,NaN,NaN,NaN
2056,NaN,NaN,NaN,NaN,NaN
2057,NaN,NaN,NaN,NaN,NaN
2058,NaN,NaN,NaN,NaN,NaN
2059,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 5. Расчёт ошибок и RMSE
stations = ['Station_A', 'Station_B', 'Station_C', 'Station_D']
for s in stations:
    df[f'Error_{s}'] = df[s] - df['Forecast']
print("Результаты RMSE:")
rmse_values = {}
for s in stations:
    rmse = np.sqrt(np.mean(df[f'Error_{s}']**2))
    rmse_values[s] = rmse
    print(f"{s}: {rmse:.4f}")

Результаты RMSE:
Station_A: 1.9618
Station_B: 1.8875
Station_C: 2.0069
Station_D: 1.9854


In [ ]:
# 6. интерактивный график 1
fig1 = go.Figure()
colors = {'Station_A': '#AA11BB', 'Station_B': 'red', 'Station_C': 'green', 'Station_D': 'orange'}
for s in stations:
    fig1.add_trace(go.Scatter(
        x=df.index,
        y=df[f'Error_{s}'],
        mode='lines+markers',
        name=s,
        line=dict(color=colors.get(s, 'blue'), width=2),
        marker=dict(size=3),
        hovertemplate=f'<b>{s}</b><br>Индекс: %{{x}}<br>Ошибка: %{{y:.4f}}<extra></extra>'
    ))
fig1.update_layout(
    title=f'Ошибки прогноза для четырёх СЭС (RMSE: A={rmse_values["Station_A"]:.3f}, B={rmse_values["Station_B"]:.3f}, C={rmse_values["Station_C"]:.3f}, D={rmse_values["Station_D"]:.3f})',
    xaxis_title='Временной индекс (часы)',
    yaxis_title='Ошибка (факт - прогноз)',
    hovermode='closest',
    width=1000,
    height=500,
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)')
)
fig1.show()


In [ ]:
# 7. интерактивный график 2
fig2 = make_subplots(
    rows=4, cols=1,
    subplot_titles=[f'{s} (RMSE = {rmse_values[s]:.4f})' for s in stations],
    shared_xaxes=True,
    vertical_spacing=0.1
)
for i, s in enumerate(stations, start=1):
    fig2.add_trace(
        go.Scatter(
            x=df.index,
            y=df[f'Error_{s}'],
            mode='lines+markers',
            name=s,
            line=dict(color=colors.get(s, 'blue'), width=2),
            marker=dict(size=2),
            showlegend=False,
            hovertemplate=f'<b>{s}</b><br>Индекс: %{{x}}<br>Ошибка: %{{y:.4f}}<extra></extra>'
        ),
        row=i, col=1
    )
    # горизонтальная линия нуля для наглядности
    fig2.add_hline(y=0, line_dash="dash", line_color="gray", row=i, col=1)
fig2.update_layout(
    title='Ошибки прогноза по каждой СЭС (отдельные графики)',
    height=900,
    width=1000,
    showlegend=False
)
fig2.update_xaxes(title_text="Временной индекс (часы)", row=4, col=1)
fig2.update_yaxes(title_text="Ошибка", row=2, col=1)
fig2.show()


In [ ]:
#8. Сохраняем интерактивные графики как HTML
pio.write_html(fig1, 'errors_combined.html')
pio.write_html(fig2, 'errors_separate.html')
# Сохраняем таблицу с ошибками
output_df = df[['Forecast'] + stations + [f'Error_{s}' for s in stations]]
output_df.to_csv('errors_forecast.csv', index=False, encoding='utf-8-sig')
output_df.to_excel('errors_forecast.xlsx', index=False, engine='openpyxl')